In [1]:
import time

from selenium import webdriver

#импортируем класс By , который позволяет выбрать способ поиска элемента
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import Select
from bs4 import BeautifulSoup

import pandas as pd


In [ ]:
#инициализация драйвера браузера.
driver = webdriver.Chrome()

#С помощью метода GET мы скажем браузеру, что надо открыть
driver.get("https://touristed.ru/toursearch")
time.sleep(10)

departure_button = driver.find_element(By.CSS_SELECTOR, ".fancybox-button.fancybox-close-small")
departure_button.click()
time.sleep(5)



The chromedriver version (135.0.7049.114) detected in PATH at C:\chromedriver\chromedriver.exe might not be compatible with the detected chrome version (136.0.7103.93); currently, chromedriver 136.0.7103.92 is recommended for chrome 136.*, so it is advised to delete the driver in PATH and retry


Самостоятельно на сайте выбираем город вылета, даты, кол-во ночей и тип сортировки.
Раскрываем все отели (кнопка внизу больше...)

Начинаем парсить сайт

In [41]:
night_button = driver.find_element(By.CSS_SELECTOR, "div.TVResultItemTitle a[data-tvtourlink]").get_attribute("href")
print(night_button)

# element = driver.find_element(By.CSS_SELECTOR, "div.TWModelItemInfo a.TWModelItemItem[target='_blank']")
# # Извлекаем href
# href_value = element.get_attribute("href")
# print("Ссылка:", href_value)


https://tourcart.ru/hotel?cd=98899949#!/hotel=ibis-phuket-kata


In [4]:
# Создаем список для хранения данных
hotel_data = []

# Ожидаем загрузки карточек
info = WebDriverWait(driver, 30).until(
    EC.presence_of_all_elements_located((By.CLASS_NAME, "TVResultListViewItem"))
)
time.sleep(10)

for inf in info:
    try:
        # Прокручиваем к элементу
        driver.execute_script("arguments[0].scrollIntoView({block: 'center', behavior: 'smooth'});", inf)
        time.sleep(10)
        
        # Создаем словарь для хранения данных текущего отеля
        hotel = {
            'name': None,
            'rating': None,
            'price': None,
            'description': None,
            'region': None,
            'href': None,
            'tours': []  # Список для хранения всех вариантов туров
        }
        
        # Основные данные отеля
        try:
            hotel['name'] = inf.find_element(By.CLASS_NAME, "TVHotelInfoTitleLink").text
            hotel['rating'] = inf.find_element(By.CLASS_NAME, "TVResultItemBeforeDescription").text
            hotel['price'] = inf.find_element(By.CLASS_NAME, "TVResultItemPriceValue").text
            hotel['description'] = inf.find_element(
                By.CSS_SELECTOR, ".TVResultItemDescription.TVLineClampEnabled.TVLineClamp-M").text
            hotel['region'] = inf.find_element(By.CLASS_NAME, "TVResultItemSubTitle").text
            hotel['href'] = inf.find_element(By.CSS_SELECTOR, "div.TVResultItemTitle a[data-tvtourlink]").get_attribute("href")          
        except NoSuchElementException as e:
            print(f"Ошибка в основных данных отеля: {str(e)}")
            continue

        # Поиск всех вариантов туров
        try:

            try:
                arrow = inf.find_element(By.CSS_SELECTOR, ".TVResultItemPrice .TVResultItemPriceValueArrow")
                driver.execute_script("arguments[0].click();", arrow)
                time.sleep(3)  # Даем время для раскрытия списка
            except NoSuchElementException:
                print("Не найдена стрелка для раскрытия списка туров")


            # Находим контейнер с турами
            tours_container = inf.find_element(
                By.CSS_SELECTOR, ".TVHotelResultItem .TVHotelResulItemTours")
            
            # Находим все строки с вариантами туров
            tour_rows = tours_container.find_elements(
                By.CSS_SELECTOR, "t-tr.TVTourResultItem")
            
            for row in tour_rows:
                try:
                    tour_info = {
                        'touroperator': row.find_element(
                            By.CSS_SELECTOR, "div.TVTourResultItemOperator").get_attribute("title"),
                        'date_tour': row.find_element(
                            By.CSS_SELECTOR, ".TVTourResultItemDate").get_attribute("textContent"),
                        'nights': row.find_element(
                            By.CSS_SELECTOR, ".TVTourResultItemNights").get_attribute("textContent"),
                        'type_room': row.find_element(
                            By.CSS_SELECTOR, "t-span.TVTourResultItemRoom").get_attribute("textContent"),
                        'meal': row.find_element(
                            By.CSS_SELECTOR, ".TVTourResultItemMeal").get_attribute("textContent"),
                        'full_price': row.find_element(
                            By.CSS_SELECTOR, ".TVTourResultItemPriceValue").get_attribute("textContent")+ " " + 
                            row.find_element(
                                By.CSS_SELECTOR, ".TVTourResultItemPriceCurrency").get_attribute("textContent")
                    }
                    hotel['tours'].append(tour_info)
                    
                except NoSuchElementException as e:
                    print(f"Не удалось извлечь некоторые данные тура: {str(e)}")
                    continue
                
        except NoSuchElementException:
            print("Не найдены варианты туров для отеля")
            hotel['tours'] = []

        # Добавляем отель в список
        hotel_data.append(hotel)

    except Exception as e:
        print(f"Критическая ошибка при обработке карточки: {str(e)}")
        continue

# Создаем расширенный DataFrame с отдельными строками для каждого варианта тура
expanded_data = []
for hotel in hotel_data:
    if not hotel['tours']:
        # Добавляем отель даже без туров
        expanded_data.append({
            'name': hotel['name'],
            'rating': hotel['rating'],
            'price': hotel['price'],
            'description': hotel['description'],
            'region': hotel['region'],
            'href': hotel['href'],
            'touroperator': 'Нет данных',
            'date_tour': 'Нет данных',
            'nights': 'Нет данных',
            'type_room': 'Нет данных',
            'meal': 'Нет данных',
            'full_price': 'Нет данных'
        })
    else:
        for tour in hotel['tours']:
            expanded_data.append({
                'name': hotel['name'],
                'rating': hotel['rating'],
                'price': hotel['price'],
                'description': hotel['description'],
                'region': hotel['region'],
                'href': hotel['href'],
                'touroperator': tour.get('touroperator', 'Нет данных'),
                'date_tour': tour.get('date_tour', 'Нет данных'),
                'nights': tour.get('nights', 'Нет данных'),
                'type_room': tour.get('type_room', 'Нет данных'),
                'meal': tour.get('meal', 'Нет данных'),
                'full_price': tour.get('full_price', 'Нет данных')
            })

# Создаем DataFrame
df = pd.DataFrame(expanded_data)
df

,name,rating,price,description,region,href,touroperator,date_tour,nights,type_room,meal,full_price
0,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Lets Fly Online,8 сен,7 нч,стандартный номер,Без питания,157 152 РУБ
1,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,TezTour,5 сен,6 нч,standard room,BB - Только завтрак,158 629 РУБ
2,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Турплатформа,8 сен,9 + 1 нч,standard room,BB - Только завтрак,161 606 РУБ
3,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Lets Fly Online,8 сен,7 нч,стандартный номер,BB - Только завтрак,162 097 РУБ
4,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Lets Fly,10 сен,7 + 1 нч,standard room,BB - Только завтрак,163 031 РУБ
...,...,...,...,...,...,...,...,...,...,...,...,...
4113,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,8 сен,6 нч,social cosy,BB - Только завтрак,275 301 РУБ
4114,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,9 сен,6 нч,social cosy,BB - Только завтрак,275 301 РУБ
4115,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,7 сен,6 нч,social cosy,BB - Только завтрак,276 672 РУБ
4116,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,6 сен,7 нч,social,BB - Только завтрак,278 901 РУБ


In [ ]:
#Извлекаем название отеля и звезду отеля
df[['hotel_name', 'star']] = df['name'].str.rsplit(n=1, expand=True)
df['star'] = df['star'].str.replace('*', '')  

#Извлекаем и преобразовываем стоимость. Оставляем валюту в отдельно столбце
df[['price_hotel', 'currency']] = df['full_price'].str.rsplit(n=1, expand=True)
df['price_hotel'] = df['price_hotel'].str.replace(' ', '').astype(float)

#Извлекаем район, город и сколько метров до моря
extracted = df['region'].str.extract(r'^(.*?),\s*(.*?)(?:,\s*(.*?)\s*до моря)?$')
df['district'] = extracted[0]
df['city'] = extracted[1]
df['meters_sea'] = extracted[2].fillna('Нет данных')



df

,name,rating,price,description,region,href,touroperator,date_tour,nights,type_room,meal,full_price,hotel_name,star,price_hotel,currency,district,city,meters_sea
0,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Lets Fly Online,2025-01-08,7 нч,стандартный номер,Без питания,157 152 РУБ,Ibis Phuket Kata,3,157152.0,РУБ,Ката,Пхукет,300 м
1,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,TezTour,2025-01-05,6 нч,standard room,BB - Только завтрак,158 629 РУБ,Ibis Phuket Kata,3,158629.0,РУБ,Ката,Пхукет,300 м
2,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Турплатформа,2025-01-08,9 + 1 нч,standard room,BB - Только завтрак,161 606 РУБ,Ibis Phuket Kata,3,161606.0,РУБ,Ката,Пхукет,300 м
3,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Lets Fly Online,2025-01-08,7 нч,стандартный номер,BB - Только завтрак,162 097 РУБ,Ibis Phuket Kata,3,162097.0,РУБ,Ката,Пхукет,300 м
4,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Lets Fly,2025-01-10,7 + 1 нч,standard room,BB - Только завтрак,163 031 РУБ,Ibis Phuket Kata,3,163031.0,РУБ,Ката,Пхукет,300 м
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4113,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,2025-01-08,6 нч,social cosy,BB - Только завтрак,275 301 РУБ,M Social Hotel Phuket (Ex. Millennium Resort P...,5,275301.0,РУБ,Патонг,Пхукет,550 м
4114,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,2025-01-09,6 нч,social cosy,BB - Только завтрак,275 301 РУБ,M Social Hotel Phuket (Ex. Millennium Resort P...,5,275301.0,РУБ,Патонг,Пхукет,550 м
4115,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,2025-01-07,6 нч,social cosy,BB - Только завтрак,276 672 РУБ,M Social Hotel Phuket (Ex. Millennium Resort P...,5,276672.0,РУБ,Патонг,Пхукет,550 м
4116,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,2025-01-06,7 нч,social,BB - Только завтрак,278 901 РУБ,M Social Hotel Phuket (Ex. Millennium Resort P...,5,278901.0,РУБ,Патонг,Пхукет,550 м


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4118 entries, 0 to 4117
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   name          4118 non-null   object        
 1   rating        4118 non-null   object        
 2   price         4118 non-null   object        
 3   description   4118 non-null   object        
 4   region        4118 non-null   object        
 5   href          4118 non-null   object        
 6   touroperator  4118 non-null   object        
 7   date_tour     4118 non-null   datetime64[ns]
 8   nights        4118 non-null   object        
 9   type_room     4118 non-null   object        
 10  meal          4118 non-null   object        
 11  full_price    4118 non-null   object        
 12  hotel_name    4118 non-null   object        
 13  star          4118 non-null   object        
 14  price_hotel   4118 non-null   float64       
 15  currency      4118 non-null   object  

In [48]:
df_1=df
df_1

,name,rating,price,description,region,href,touroperator,date_tour,nights,type_room,meal,full_price,hotel_name,star,price_hotel,currency,district,city,meters_sea
0,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Lets Fly Online,2025-01-08,7 нч,стандартный номер,Без питания,157 152 РУБ,Ibis Phuket Kata,3,157152.0,РУБ,Ката,Пхукет,300 м
1,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,TezTour,2025-01-05,6 нч,standard room,BB - Только завтрак,158 629 РУБ,Ibis Phuket Kata,3,158629.0,РУБ,Ката,Пхукет,300 м
2,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Турплатформа,2025-01-08,9 + 1 нч,standard room,BB - Только завтрак,161 606 РУБ,Ibis Phuket Kata,3,161606.0,РУБ,Ката,Пхукет,300 м
3,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Lets Fly Online,2025-01-08,7 нч,стандартный номер,BB - Только завтрак,162 097 РУБ,Ibis Phuket Kata,3,162097.0,РУБ,Ката,Пхукет,300 м
4,Ibis Phuket Kata 3*,4.0,157 152,Уютный современный отель с удачным расположени...,"Ката, Пхукет, 300 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Lets Fly,2025-01-10,7 + 1 нч,standard room,BB - Только завтрак,163 031 РУБ,Ibis Phuket Kata,3,163031.0,РУБ,Ката,Пхукет,300 м
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4113,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,2025-01-08,6 нч,social cosy,BB - Только завтрак,275 301 РУБ,M Social Hotel Phuket (Ex. Millennium Resort P...,5,275301.0,РУБ,Патонг,Пхукет,550 м
4114,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,2025-01-09,6 нч,social cosy,BB - Только завтрак,275 301 РУБ,M Social Hotel Phuket (Ex. Millennium Resort P...,5,275301.0,РУБ,Патонг,Пхукет,550 м
4115,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,2025-01-07,6 нч,social cosy,BB - Только завтрак,276 672 РУБ,M Social Hotel Phuket (Ex. Millennium Resort P...,5,276672.0,РУБ,Патонг,Пхукет,550 м
4116,M Social Hotel Phuket (Ex. Millennium Resort P...,4.7,266 644,Окунитесь в особый ритм Патонга в отеле с уютн...,"Патонг, Пхукет, 550 м до моря",https://tourcart.ru/hotel?cd=98899948#!/hotel=...,Biblioglobus,2025-01-06,7 нч,social,BB - Только завтрак,278 901 РУБ,M Social Hotel Phuket (Ex. Millennium Resort P...,5,278901.0,РУБ,Патонг,Пхукет,550 м


In [24]:
import sqlite3
#создаем новую базу данных sqlite 
con_sqlite = sqlite3.connect('travel.db')

In [26]:
#загружаем датафрейм в базу данных sqlite в new_db_weather
df.to_sql(con=con_sqlite, name = 'tailand',if_exists='replace')

4118